**Name:** Aashika Joshi
**Roll Number:** 1024170209

## Q1: Build Your Personalized Knowledge Base

Take your college roll number, extract its digits, and build a pandas DataFrame with 6 FAQ entries — 4 fixed ones plus 2 built from the last two digits of the roll number.

In [2]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

roll_no = "1024170209"
digits = [int(d) for d in roll_no]
print("Roll number:", roll_no)
print("Digits:", digits)

last_two = roll_no[-2:]
categories = ["billing", "account", "general"]

for d in last_two:
    d = int(d)
    print(f"digit {d} -> category[{d} % 3] = {categories[d % 3]}")

fixed_faqs = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

personal_faqs = [
    {"question": "how do i update my registered mobile number",
     "answer": "Go to Profile > Contact Details and update your mobile number.",
     "keywords": "mobile number update contact", "category": "account"},
    {"question": "how do i contact customer support",
     "answer": "You can email support@college.edu or call our helpline.",
     "keywords": "contact support help", "category": "general"},
]

faq_entries = fixed_faqs + personal_faqs
df = pd.DataFrame(faq_entries)
df

Roll number: 1024170209
Digits: [1, 0, 2, 4, 1, 7, 0, 2, 0, 9]
digit 0 -> category[0 % 3] = billing
digit 9 -> category[9 % 3] = billing


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do i update my registered mobile number,Go to Profile > Contact Details and update you...,mobile number update contact,account
5,how do i contact customer support,You can email support@college.edu or call our ...,contact support help,general


## Q2: Generate and Score a Hypothesis

A scoring function that takes a query string and returns all matching entries ranked by confidence. A keyword match counts more than a plain question-word match.

In [3]:
def find_matching_faq(query, df):
    query_terms = set(query.lower().split())
    matches = []

    for _, row in df.iterrows():
        faq_keywords = set(row["keywords"].lower().split())
        faq_question = set(row["question"].lower().split())

        keyword_score = len(query_terms & faq_keywords) * 2
        question_score = len(query_terms & faq_question)

        total_score = keyword_score + question_score

        if total_score > 0:
            matches.append(
                (
                    row["question"],
                    row["answer"],
                    total_score
                )
            )

    matches.sort(key=lambda item: item[2], reverse=True)

    return matches


search_query = "how can i make a payment"

print("Search Query:", search_query)
print("\nMatching FAQs:")

for question, answer, score in find_matching_faq(search_query, df):
    print(f"Score {score}: {question}")
    print(f"Answer: {answer}")
    print()

Search Query: how can i make a payment

Matching FAQs:
Score 5: how can i pay the fee
Answer: You can pay via UPI, card, or net banking.

Score 2: how do i update my registered mobile number
Answer: Go to Profile > Contact Details and update your mobile number.

Score 2: how do i contact customer support
Answer: You can email support@college.edu or call our helpline.

Score 1: how to reset password
Answer: Go to Settings > Reset Password.



## Q3: Questions By Category

`same_category(category_name, df)` returns all questions belonging to a given category. Calling it with `"account"`, the category of one of the personalized entries from Q1.

In [4]:
def same_category(category_name, df):
    filtered_data = df[df["category"] == category_name]
    return filtered_data["question"].tolist()


selected_category = "billing"

print("Selected Category:", selected_category)
print("\nQuestions in this category:")

print(same_category(selected_category, df))

Selected Category: billing

Questions in this category:
['what is the annual fee', 'how can i pay the fee']


## Q4: Add a Keyword and Save to CSV

Pick one entry, add a new keyword typed by the user, and save the whole updated DataFrame to `1024170209_faq_data.csv`.

In [9]:
new_keyword = input("Enter a new keyword: ")

target_question = "what is the yearly college fee"

df.loc[
    df["question"] == target_question,
    "keywords"
] = (
    df.loc[
        df["question"] == target_question,
        "keywords"
    ] + " " + new_keyword
)

output_file = "1024170209_faq_data.csv"

df.to_csv(output_file, index=False)

print("Keyword added successfully:", new_keyword)
print("DataFrame saved as:", output_file)

df

Enter a new keyword:  abcdd


Keyword added successfully: abcdd
DataFrame saved as: 1024170209_faq_data.csv


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do i update my registered mobile number,Go to Profile > Contact Details and update you...,mobile number update contact,account
5,how do i contact customer support,You can email support@college.edu or call our ...,contact support help,general


## Q5: FAQ Entries Per Category

Using `groupby` to count how many FAQ entries fall under each category.

In [8]:
faq_summary = df.groupby("category").size()

print("Number of FAQs in each category:")
print(faq_summary)

Number of FAQs in each category:
category
account    2
billing    2
general    2
dtype: int64


## Q6: Handling Ties in the Scoring Function

The scoring function is updated so that when two or more entries tie for the highest score, all of them are printed instead of silently picking one. One query below produces a tie, and one does not.

In [10]:
def find_best_faqs(query, df):
    query_terms = set(query.lower().split())
    matches = []

    for _, row in df.iterrows():
        faq_keywords = set(row["keywords"].lower().split())
        faq_question = set(row["question"].lower().split())

        score = (
            len(query_terms & faq_keywords) * 2
            + len(query_terms & faq_question)
        )

        if score > 0:
            matches.append(
                (
                    row["question"],
                    row["answer"],
                    score
                )
            )

    if not matches:
        return []

    highest_score = max(item[2] for item in matches)

    return [
        item for item in matches
        if item[2] == highest_score
    ]


tie_query = "how"

print("Query 1:", tie_query)
print("Top matching entries:")

for question, answer, score in find_best_faqs(tie_query, df):
    print(score, "-", question, "-", answer)


print("\n" + "-" * 50 + "\n")


single_query = "reset password"

print("Query 2:", single_query)
print("Top matching entry:")

for question, answer, score in find_best_faqs(single_query, df):
    print(score, "-", question, "-", answer)

Query 1: how
Top matching entries:
1 - how to reset password - Go to Settings > Reset Password.
1 - how can i pay the fee - You can pay via UPI, card, or net banking.
1 - how do i update my registered mobile number - Go to Profile > Contact Details and update your mobile number.
1 - how do i contact customer support - You can email support@college.edu or call our helpline.

--------------------------------------------------

Query 2: reset password
Top matching entry:
6 - how to reset password - Go to Settings > Reset Password.
